# Exploratory Data Analysis - Customer Support Tickets

This notebook performs exploratory data analysis on customer support tickets.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing.data_loader import DataLoader
from src.preprocessing.text_processor import TextPreprocessor
from config.config import PREPROCESSING_CONFIG

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data

In [ ]:
# Load data
loader = DataLoader()

# Try to load Twitter support data (will create sample if not available)
df = loader.load_twitter_support()

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 2. Basic Statistics

In [ ]:
# Basic info
print("Dataset Info:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDataset description:")
print(df.describe())

## 3. Label Distribution

In [ ]:
# Label distribution
label_counts = df['label'].value_counts()
print("Label distribution:")
print(label_counts)
print(f"\nNumber of unique labels: {df['label'].nunique()}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot
label_counts.plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Ticket Category Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Category', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.tick_params(axis='x', rotation=45)

# Pie chart
label_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%')
ax2.set_title('Ticket Category Percentage', fontsize=14, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

# Class imbalance ratio
max_count = label_counts.max()
min_count = label_counts.min()
print(f"\nClass imbalance ratio: {max_count / min_count:.2f}:1")

## 4. Text Length Analysis

In [ ]:
# Calculate text lengths
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print("Text length statistics:")
print(df[['text_length', 'word_count']].describe())

# Plot distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Character length distribution
ax1.hist(df['text_length'], bins=50, color='skyblue', edgecolor='black')
ax1.set_title('Distribution of Text Length (Characters)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Character Count', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.axvline(df['text_length'].mean(), color='red', linestyle='--', label=f'Mean: {df["text_length"].mean():.0f}')
ax1.legend()

# Word count distribution
ax2.hist(df['word_count'], bins=50, color='lightgreen', edgecolor='black')
ax2.set_title('Distribution of Word Count', fontsize=14, fontweight='bold')
ax2.set_xlabel('Word Count', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.axvline(df['word_count'].mean(), color='red', linestyle='--', label=f'Mean: {df["word_count"].mean():.0f}')
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Text Length by Category

In [ ]:
# Box plot of text length by category
plt.figure(figsize=(12, 6))
df.boxplot(column='text_length', by='label', figsize=(14, 6))
plt.title('Text Length Distribution by Category', fontsize=14, fontweight='bold')
plt.suptitle('')  # Remove default title
plt.xlabel('Category', fontsize=12)
plt.ylabel('Character Count', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Average text length by category
avg_length = df.groupby('label')['text_length'].mean().sort_values(ascending=False)
print("\nAverage text length by category:")
print(avg_length)

## 6. Word Clouds by Category

In [ ]:
# Create word clouds for each category
categories = df['label'].unique()
n_categories = len(categories)
n_cols = 3
n_rows = (n_categories + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
axes = axes.flatten() if n_categories > 1 else [axes]

for idx, category in enumerate(categories):
    if idx >= len(axes):
        break

    # Get texts for this category
    texts = ' '.join(df[df['label'] == category]['text'].astype(str))

    # Generate word cloud
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='viridis',
        max_words=100
    ).generate(texts)

    # Plot
    axes[idx].imshow(wordcloud, interpolation='bilinear')
    axes[idx].set_title(f'{category}', fontsize=14, fontweight='bold')
    axes[idx].axis('off')

# Hide empty subplots
for idx in range(n_categories, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 7. Sample Texts by Category

In [ ]:
# Show sample texts for each category
print("Sample tickets by category:\n")
for category in df['label'].unique():
    print(f"\n{'='*80}")
    print(f"Category: {category}")
    print(f"{'='*80}")

    samples = df[df['label'] == category]['text'].head(3)
    for i, text in enumerate(samples, 1):
        print(f"\n[{i}] {text}")
    print()

## 8. Data Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = TextPreprocessor(**PREPROCESSING_CONFIG)

# Example: Clean a sample text
sample_text = df['text'].iloc[0]
print("Original text:")
print(sample_text)
print("\nCleaned text:")
print(preprocessor.clean_text(sample_text))

## 9. Train/Val/Test Split

In [ ]:
# Split data
train_df, val_df, test_df = loader.prepare_dataset(df)

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

# Check distribution in each split
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, data) in zip(axes, [('Train', train_df), ('Validation', val_df), ('Test', test_df)]):
    data['label'].value_counts().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'{name} Set Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Category', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 10. Save Processed Data

In [ ]:
# Preprocess texts
train_df['text'] = preprocessor.process(train_df['text'].tolist())
val_df['text'] = preprocessor.process(val_df['text'].tolist())
test_df['text'] = preprocessor.process(test_df['text'].tolist())

# Save processed data
loader.save_processed_data(train_df, val_df, test_df)

print("Processed data saved!")

## Key Findings

1. **Class Distribution**: [Analyze if balanced or imbalanced]
2. **Text Length**: [Average length and variation]
3. **Category Characteristics**: [Key differences between categories]
4. **Next Steps**: Ready for baseline model training